In [ ]:
import flax.linen as nn
import optax
import h5py as hf
import matplotlib.pyplot as plt
import znnl as nl
import numpy as np
from znnl.data.decision_boundary import (
    DecisionBoundaryGenerator,
    circle,
    linear_boundary,
)
# Dask support
from dask.distributed import Client
from dask_jobqueue import SLURMCluster

In [ ]:
ds_size=500

In [ ]:
def train(index: int):
    """
    Run the experiment.
    """
    print("Starting")

    class Network(nn.Module):
        """
        Perceptron network.
        """

        @nn.compact
        def __call__(self, x):
            """
            Call method for the network
            """
            x = nn.Dense(100, use_bias=False)(x)
            x = nn.relu(x)
            x = nn.Dense(100, use_bias=False)(x)
            x = nn.relu(x)
            x = nn.Dense(1, use_bias=False)(x)
            return nn.sigmoid(x)
        
    generator = DecisionBoundaryGenerator(ds_size, discriminator="circle",  one_hot=False)
    model = nl.models.FlaxModel(
        flax_module=Network(), 
        optimizer=optax.sgd(1.0), 
        input_shape=(1, 2)
    )
    # Prepare the recorders
    train_recorder = nl.training_recording.JaxRecorder(
        name=f"two-layer-100/train_recorder_{index}",
        loss=True,
        entropy=True,
        trace=True,
        accuracy=True,
        magnitude_variance=True,
        update_rate=1,
    )
    test_recorder = nl.training_recording.JaxRecorder(
        name=f"two-layer-100/test_recorder_{index}", loss=True, accuracy=True, update_rate=1,
    )
    train_recorder.instantiate_recorder(data_set=generator.train_ds)
    test_recorder.instantiate_recorder(data_set=generator.test_ds)

    trainer = nl.training_strategies.SimpleTraining(
        model=model,
        loss_fn=nl.loss_functions.MeanPowerLoss(order=2),
        accuracy_fn=nl.accuracy_functions.LabelAccuracy(),
        recorders=[train_recorder, test_recorder],
    )

    _ = trainer.train_model(
        train_ds=generator.train_ds,
        test_ds=generator.test_ds,
        batch_size=125,
        epochs=500,
    )


## Deployment

In [ ]:
# def deploy_jobs(index):
cluster = SLURMCluster(
    cores=1,
    processes=1,
    job_cpu=30,
    memory="64GB",
    queue="single",
    walltime="12:00:00",
    log_directory=f'./two-layer-100/dask-logs',
    job_script_prologue=["module load devel/cuda/12.1"],
    job_extra_directives=["--gres=gpu:2"]
)
cluster.adapt(minimum_jobs=0, maximum_jobs=20)
client = Client(cluster)

In [ ]:
results = []
clients = []
clusters = []

# indice_blocks = np.linspace(1, 20, # 20, dtype=int)
indice_blocks = [13, 14, 15, 16, 17, 18, 19, 20]

results = client.map(train, indice_blocks)


In [ ]:
results

In [ ]:
client.close()
cluster.close()